In [1]:
!nvidia-smi

Mon Jun  3 20:29:03 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.171.04             Driver Version: 535.171.04   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:01:00.0 Off |                  N/A |
|  0%   49C    P5             118W / 370W |      1MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import resnet50
from torch.nn import Transformer

# تعریف مدل
class TransformerModel(nn.Module):
    def __init__(self):
        super(TransformerModel, self).__init__()
        self.encoder = resnet50(pretrained=True)
        self.transformer = Transformer(d_model=2048, nhead=8, num_encoder_layers=6, num_decoder_layers=6)
        self.decoder = nn.Linear(2048, 2)  # دو کلاس برای پیش‌بینی: تومور و غیر تومور

    def forward(self, x):
        x = self.encoder(x)
        x = self.transformer(x)
        x = self.decoder(x)
        return x

# آماده‌سازی داده‌ها
#transform = transforms.Compose([
#    transforms.Resize((256, 256)),
#    transforms.ToTensor(),
#])


In [3]:
import torch
from torch import nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer

class ImageSegmentationModel(nn.Module):
    def __init__(self, num_classes, d_model=512, nhead=8, num_layers=6):
        super().__init__()

        # Transformer backbone
        encoder_layers = TransformerEncoderLayer(d_model, nhead)
        self.transformer_encoder = TransformerEncoder(encoder_layers, num_layers)

        # Segmentation head
        self.segmentation_head = nn.Conv2d(d_model, num_classes, kernel_size=1)

    def forward(self, x):
        # Reshape input for transformer (sequence length, batch size, embedding dimension)
        x = x.flatten(2).permute(2, 0, 1)

        # Pass through transformer
        x = self.transformer_encoder(x)

        # Reshape output from transformer for segmentation head
        x = x.permute(1, 2, 0).unsqueeze(-1)

        # Pass through segmentation head
        x = self.segmentation_head(x)

        return x


In [4]:
# preprocessing
train_transform = transforms.Compose([
    transforms.Resize([224,224]),
    transforms.Lambda(lambda image: image.convert('RGB')),
    #transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    #transforms.GaussianBlur(11, sigma=(0.1, 2.0)),
    #torchvision.transforms.AugMix(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])
test_transform = transforms.Compose([
    transforms.Resize([224,224]),
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])


In [1]:
import albumentations as A

def get_train_augs():
  return A.Compose([
      A.Resize(IMAGE_SIZE, IMAGE_SIZE),
      A.HorizontalFlip(p=0.5),
      A.VerticalFlip(p=0.5)
  ])
def get_valid_augs():
  return A.Compose([
      A.Resize(IMAGE_SIZE, IMAGE_SIZE)
    ])

ModuleNotFoundError: No module named 'albumentations'

In [ ]:
from torch.utils.data import Dataset

class SegmentationDataset(Dataset):
  def __init__(self,df, augmentations):
    self.df=df
    self.augmentations = augmentations 
  def __len__(self):
    return len(self.df)
  def __getitem__(self ,idx):
    row = self.df.iloc[idx]
    image_path = row.images
    mask_path = row.masks

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(mask_path , cv2.IMREAD_GRAYSCALE) #(h,w,c)
    mask =np.expand_dims(mask ,axis=-1)

    if self.augmentations :
      data =self.augmentations(image = image , mask = mask)
      image = data['image']
      mask = data ['mask']

    #(h,w,c) -> (c,h,w)
    image = np.transpose(image ,(2,0,1)).astype(np.float32)
    mask = np.transpose(mask ,(2,0,1)).astype(np.float32)

    image = torch.Tensor(image)/255.0
    mask = torch.round(torch.Tensor(mask)/255.0)

    return image,mask

In [6]:
import torch

class BinaryJaccardLoss(torch.nn.Module):
    def forward(self, pred, target):
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum() - intersection
        jaccard_index = intersection / union
        return 1 - jaccard_index


In [7]:
import os
from PIL import Image
from torch.utils.data import Dataset, random_split, DataLoader
from torchvision.transforms import ToTensor

class SegmentationDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(img_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        return image, mask

# Define the directories for your images and masks
img_dir = 'Dataset/train/Flair'
mask_dir = 'Dataset/train/Mask'

# Create the dataset
dataset = SegmentationDataset(img_dir, mask_dir, transform=ToTensor())

# Define the split for train and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

# Split the dataset
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


In [8]:
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from copy import deepcopy
import numpy as np



# تقسیم داده‌ها به آموزش و ولیدیشن
#train_dataset, val_dataset = train_test_split(dataset1, test_size=0.2)
train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)

print(f"total no,of batches in trainloader: {len(train_dataloader)}")
print(f"total no,of batches in validloader: {len(val_dataloader)}")

for image , mask in train_dataloader:
  break
print (f'One batch image shape: {image.shape}')
print (f'One batch image shape: {mask.shape}')

# تعریف early stopping
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), 'checkpoint.pt')
        self.val_loss_min = val_loss

early_stopping = EarlyStopping(patience=7, verbose=True)


total no,of batches in trainloader: 8649
total no,of batches in validloader: 2163
One batch image shape: torch.Size([1, 3, 182, 182])
One batch image shape: torch.Size([1, 1, 182, 182])


In [9]:
model = ImageSegmentationModel(num_classes=2, d_model=3, nhead=1, num_layers=6)
#model = TransformerModel()

#model = model.cuda()

from torchsummary import summary
summary(model, (3, 182, 182))

/home/jupyter-ziaratban_stu4/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat2 in method wrapper_CUDA_mm)

In [ ]:
import GPUtil

# آموزش مدل با early stopping


num_epochs = 100

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, 'min')

for epoch in range(num_epochs):
    GPUtil.showUtilization()

    model.train()
    for i, (images, masks) in enumerate(train_dataloader):
        images, masks = images.cuda(), masks.cuda()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = 0
        for i, (images, masks) in enumerate(val_dataloader):
            images, masks = images.cuda(), masks.cuda()
            outputs = model(images)
            val_loss += criterion(outputs, masks).item()
        val_loss /= len(val_dataloader)

    scheduler.step(val_loss)
    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print("Early stopping")
        break

# بارگذاری مدل بهترین
model.load_state_dict(torch.load('checkpoint.pt'))
